## Causal self attention mechanism

In [1]:
import numpy as np
import torch
import math

In [3]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

In [4]:
x_2 = inputs.shape[1]

# input dimensions
dimension_input = inputs.shape[1]

dimension_output = 2

In [6]:
torch.manual_seed(123)

weight_query = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_key = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_value = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)

In [7]:
# Compute Q
queries = inputs @ weight_query

In [8]:
# compute K
keys = inputs @ weight_key

In [9]:
# compute V
values = inputs @ weight_value

In [10]:
# compute attention scores
attention_scores = queries @ keys.T
attention_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

In [11]:
# get the dimension of keys
dimensions_keys = keys.shape[-1]

In [12]:
scaled_weights = attention_scores / math.sqrt(dimensions_keys)

In [13]:
attention_weights = torch.softmax(scaled_weights, dim=-1)
attention_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

### create the causal mask

In [14]:
context_length = inputs.shape[0]

# create a ones tensor
tensor_ones = torch.ones(context_length, context_length)
tensor_ones

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [16]:
# create the causal mask
causal_mask = torch.tril(tensor_ones)
causal_mask

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [17]:
# causal attention scores
causal_attention_scores = attention_weights * causal_mask
causal_attention_scores

tensor([[0.1551, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2264, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1503, 0.2256, 0.2192, 0.0000, 0.0000, 0.0000],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.0000, 0.0000],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

In [20]:
causal_scaled_weights = causal_attention_scores / math.sqrt(dimensions_keys)
causal_scaled_weights

tensor([[0.1097, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1061, 0.1601, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1063, 0.1595, 0.1550, 0.0000, 0.0000, 0.0000],
        [0.1125, 0.1410, 0.1388, 0.1045, 0.0000, 0.0000],
        [0.1138, 0.1378, 0.1360, 0.1061, 0.0895, 0.0000],
        [0.1101, 0.1479, 0.1448, 0.1004, 0.0770, 0.1269]])

In [21]:
causal_attention_weights = torch.softmax(causal_scaled_weights, dim=1)
causal_attention_weights

tensor([[0.1825, 0.1635, 0.1635, 0.1635, 0.1635, 0.1635],
        [0.1769, 0.1867, 0.1591, 0.1591, 0.1591, 0.1591],
        [0.1724, 0.1818, 0.1810, 0.1550, 0.1550, 0.1550],
        [0.1714, 0.1763, 0.1759, 0.1700, 0.1532, 0.1532],
        [0.1693, 0.1734, 0.1731, 0.1680, 0.1652, 0.1511],
        [0.1653, 0.1717, 0.1712, 0.1637, 0.1599, 0.1681]])